# 🏦 Análise de Performance Financeira — Deutsche Bank (2015–2017)

**Objetivo:** Analisar indicadores financeiros do Deutsche Bank ao longo de 800 registros diários (Jan/2015 a Mar/2017), identificando tendências de rentabilidade, eficiência operacional, alavancagem e risco.

**Dataset:** 800 registros | 15 indicadores financeiros | Fonte: deutsche_bank_financial_perform

**Ferramentas:** Python · Pandas · Matplotlib · Seaborn

---

## Estrutura da Análise
1. Importação e reconhecimento dos dados
2. Limpeza e preparação
3. Visão geral de receita e lucro
4. Eficiência operacional
5. Indicadores de rentabilidade (ROA e Margem de Lucro)
6. Análise de alavancagem e risco (Debt-to-Equity)
7. Fluxo de caixa e dividendos
8. Correlações entre variáveis
9. Insights e conclusões

## 1. Importação e Reconhecimento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})

AZUL    = '#1565C0'
VERDE   = '#2E7D32'
LARANJA = '#E65100'
VERMELHO= '#C62828'
CINZA   = '#546E7A'

df = pd.read_excel('banco_alemão.xlsx', sheet_name='deutsche_bank_financial_perform')
meta = pd.read_excel('banco_alemão.xlsx', sheet_name='metadados')

print(f'Shape: {df.shape}')
print(f'Período: {df["Date"].min().date()} a {df["Date"].max().date()}')
df.head(3)

In [ ]:
print('Dicionário de variáveis:')
meta

## 2. Limpeza e Preparação

In [ ]:
print('Valores nulos:')
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else '✅ Nenhum valor nulo encontrado.')

# Features temporais
df['Ano']    = df['Date'].dt.year
df['Mes']    = df['Date'].dt.month
df['AnoMes'] = df['Date'].dt.to_period('M')

# Eficiência operacional: quanto gera para cada R$1 gasto
df['Eficiencia_Op'] = df['Operating_Income'] / df['Expenses']

# Flag de prejuízo operacional
df['Prejuizo_Op'] = df['Eficiencia_Op'] < 1

print(f'\n✅ Features criadas: Ano, Mes, AnoMes, Eficiencia_Op, Prejuizo_Op')
print(f'Registros com prejuízo operacional: {df["Prejuizo_Op"].sum()} ({df["Prejuizo_Op"].mean()*100:.1f}%)')
print(f'Registros com margem de lucro negativa: {(df["Profit_Margin"]<0).sum()} ({(df["Profit_Margin"]<0).mean()*100:.1f}%)')

## 3. Visão Geral de Receita e Lucro

In [ ]:
anual = df.groupby('Ano').agg(
    Receita=('Revenue', 'sum'),
    Lucro_Liquido=('Net_Income', 'sum'),
    Despesas=('Expenses', 'sum'),
    Income_Op=('Operating_Income', 'sum')
).reset_index()

# 2017 parcial — apenas 69 dias
print('Resumo anual:')
print('(* 2017 parcial: Jan–Mar apenas)')
fmt = anual.copy()
for col in ['Receita','Lucro_Liquido','Despesas','Income_Op']:
    fmt[col] = fmt[col].map('R$ {:,.0f}'.format)
print(fmt.to_string(index=False))

In [ ]:
# Evolução mensal de receita e lucro
mensal = df.groupby('AnoMes').agg(
    Receita=('Revenue', 'sum'),
    Lucro=('Net_Income', 'sum'),
    Despesas=('Expenses', 'sum')
).reset_index()
mensal['AnoMes_dt'] = mensal['AnoMes'].dt.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Evolução Financeira Mensal — Deutsche Bank (2015–2017)', fontsize=14, fontweight='bold')

# Receita
axes[0].fill_between(mensal['AnoMes_dt'], mensal['Receita'],
                     alpha=0.15, color=AZUL)
axes[0].plot(mensal['AnoMes_dt'], mensal['Receita'],
             color=AZUL, linewidth=2, marker='o', markersize=3)
axes[0].set_ylabel('Receita (R$)')
axes[0].set_title('Receita Mensal')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))

# Lucro vs Despesas
axes[1].fill_between(mensal['AnoMes_dt'], mensal['Lucro'],
                     where=mensal['Lucro'] >= 0, alpha=0.15, color=VERDE, label='Lucro')
axes[1].fill_between(mensal['AnoMes_dt'], mensal['Lucro'],
                     where=mensal['Lucro'] < 0, alpha=0.15, color=VERMELHO, label='Prejuízo')
axes[1].plot(mensal['AnoMes_dt'], mensal['Lucro'],
             color=VERDE, linewidth=2, marker='o', markersize=3)
axes[1].axhline(0, color=VERMELHO, linestyle='--', linewidth=1)
axes[1].set_ylabel('Lucro Líquido (R$)')
axes[1].set_title('Lucro Líquido Mensal')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

plt.tight_layout()
plt.savefig('img_receita_lucro.png', bbox_inches='tight')
plt.show()
print('💡 Receita relativamente estável entre 2015 e 2016.')
print('💡 Lucro líquido apresenta alta volatilidade mês a mês — sinal de irregularidade operacional.')

## 4. Eficiência Operacional

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição da eficiência operacional
axes[0].hist(df['Eficiencia_Op'], bins=30, color=AZUL, edgecolor='white', alpha=0.85)
media_ef = df['Eficiencia_Op'].mean()
axes[0].axvline(1, color=VERMELHO, linestyle='--', linewidth=2, label='Break-even (=1)')
axes[0].axvline(media_ef, color=LARANJA, linestyle='--', linewidth=2,
                label=f'Média: {media_ef:.2f}')
axes[0].set_title('Distribuição da Eficiência Operacional\n(Income / Expenses)')
axes[0].set_xlabel('Índice de Eficiência')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Eficiência por ano (boxplot)
anos_labels = {2015: '2015\n(365 dias)', 2016: '2016\n(366 dias)', 2017: '2017\n(69 dias)*'}
df['Ano_Label'] = df['Ano'].map(anos_labels)
df.boxplot(column='Eficiencia_Op', by='Ano_Label', ax=axes[1],
           boxprops=dict(color=AZUL),
           medianprops=dict(color=VERMELHO, linewidth=2),
           whiskerprops=dict(color=CINZA),
           capprops=dict(color=CINZA))
axes[1].axhline(1, color=VERMELHO, linestyle='--', linewidth=1.5, label='Break-even')
axes[1].set_title('Eficiência Operacional por Ano')
axes[1].set_xlabel('')
axes[1].set_ylabel('Eficiência (Income / Expenses)')
axes[1].legend()
plt.suptitle('')

plt.tight_layout()
plt.savefig('img_eficiencia.png', bbox_inches='tight')
plt.show()
print(f'💡 Eficiência média de {media_ef:.2f}x — para cada R$1,00 gasto, o banco gera R${media_ef:.2f} de income operacional.')
print(f'💡 {df["Prejuizo_Op"].sum()} registros ({df["Prejuizo_Op"].mean()*100:.0f}%) com eficiência < 1 indicam dias de prejuízo operacional.')

## 5. Indicadores de Rentabilidade

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Indicadores de Rentabilidade', fontsize=14, fontweight='bold')

# ROA ao longo do tempo
roa_mensal = df.groupby('AnoMes')['ROA'].mean().reset_index()
roa_mensal['dt'] = roa_mensal['AnoMes'].dt.to_timestamp()
axes[0,0].plot(roa_mensal['dt'], roa_mensal['ROA'], color=AZUL, linewidth=2)
axes[0,0].fill_between(roa_mensal['dt'], roa_mensal['ROA'],
                       where=roa_mensal['ROA'] >= 0, alpha=0.15, color=VERDE)
axes[0,0].fill_between(roa_mensal['dt'], roa_mensal['ROA'],
                       where=roa_mensal['ROA'] < 0, alpha=0.15, color=VERMELHO)
axes[0,0].axhline(0, color=VERMELHO, linestyle='--', linewidth=1)
axes[0,0].set_title('ROA Médio Mensal')
axes[0,0].set_ylabel('Return on Assets')
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[0,0].tick_params(axis='x', rotation=30)

# Distribuição do ROA
axes[0,1].hist(df['ROA'], bins=25, color=VERDE, edgecolor='white', alpha=0.85)
axes[0,1].axvline(df['ROA'].mean(), color=VERMELHO, linestyle='--',
                  linewidth=2, label=f'Média: {df["ROA"].mean():.4f}')
axes[0,1].axvline(0, color=CINZA, linestyle='-', linewidth=1.5)
axes[0,1].set_title('Distribuição do ROA')
axes[0,1].set_xlabel('ROA')
axes[0,1].set_ylabel('Frequência')
axes[0,1].legend()

# Margem de lucro ao longo do tempo
mg_mensal = df.groupby('AnoMes')['Profit_Margin'].mean().reset_index()
mg_mensal['dt'] = mg_mensal['AnoMes'].dt.to_timestamp()
cores_mg = [VERDE if v >= 0 else VERMELHO for v in mg_mensal['Profit_Margin']]
axes[1,0].bar(mg_mensal['dt'], mg_mensal['Profit_Margin'],
              color=cores_mg, edgecolor='white', width=20)
axes[1,0].axhline(0, color='black', linewidth=1)
axes[1,0].set_title('Margem de Lucro Média Mensal')
axes[1,0].set_ylabel('Profit Margin')
axes[1,0].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[1,0].tick_params(axis='x', rotation=30)

# ROA por ano (comparativo)
roa_ano = df.groupby('Ano')['ROA'].agg(['mean','std']).reset_index()
bars = axes[1,1].bar(roa_ano['Ano'].astype(str), roa_ano['mean'],
                     yerr=roa_ano['std'], color=[AZUL, LARANJA, VERDE],
                     edgecolor='white', width=0.4,
                     error_kw={'capsize': 5, 'elinewidth': 1.5})
axes[1,1].set_title('ROA Médio por Ano (±1 desvio padrão)')
axes[1,1].set_ylabel('ROA')
axes[1,1].set_xlabel('* 2017 parcial (Jan–Mar)')
for bar in bars:
    h = bar.get_height()
    axes[1,1].text(bar.get_x() + bar.get_width()/2, h + 0.001,
                   f'{h:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('img_rentabilidade.png', bbox_inches='tight')
plt.show()
print(f'💡 ROA médio de {df["ROA"].mean():.4f} — baixo, típico de grandes bancos com base de ativos elevada.')
print(f'💡 {(df["Profit_Margin"] < 0).sum()} registros com margem negativa ({(df["Profit_Margin"]<0).mean()*100:.1f}% do período).')

## 6. Alavancagem e Risco (Debt-to-Equity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Evolução do D/E ao longo do tempo
de_mensal = df.groupby('AnoMes')['Debt_to_Equity'].mean().reset_index()
de_mensal['dt'] = de_mensal['AnoMes'].dt.to_timestamp()
axes[0].plot(de_mensal['dt'], de_mensal['Debt_to_Equity'],
             color=VERMELHO, linewidth=2.5)
axes[0].fill_between(de_mensal['dt'], de_mensal['Debt_to_Equity'],
                     alpha=0.1, color=VERMELHO)
# Referência: D/E > 5 = alto risco
axes[0].axhline(5, color=LARANJA, linestyle='--', linewidth=1.5,
                label='Alerta: D/E = 5')
axes[0].axhline(df['Debt_to_Equity'].mean(), color=CINZA, linestyle=':',
                linewidth=1.5, label=f'Média: {df["Debt_to_Equity"].mean():.2f}')
axes[0].set_title('Evolução do Debt-to-Equity Ratio')
axes[0].set_ylabel('Debt-to-Equity')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[0].tick_params(axis='x', rotation=30)

# D/E por ano
de_ano = df.groupby('Ano')['Debt_to_Equity'].agg(['mean','median']).reset_index()
x = np.arange(len(de_ano))
w = 0.35
axes[1].bar(x - w/2, de_ano['mean'], width=w, color=VERMELHO,
            edgecolor='white', label='Média', alpha=0.85)
axes[1].bar(x + w/2, de_ano['median'], width=w, color=LARANJA,
            edgecolor='white', label='Mediana', alpha=0.85)
axes[1].axhline(5, color=CINZA, linestyle='--', linewidth=1.5, label='Alerta D/E=5')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['2015', '2016', '2017*'])
axes[1].set_title('Debt-to-Equity por Ano')
axes[1].set_ylabel('Ratio')
axes[1].legend()
axes[1].set_xlabel('* 2017 parcial')

plt.tight_layout()
plt.savefig('img_alavancagem.png', bbox_inches='tight')
plt.show()
alto_de = (df['Debt_to_Equity'] > 5).sum()
print(f'💡 D/E médio de {df["Debt_to_Equity"].mean():.2f} — acima de 5 em {alto_de} registros ({alto_de/len(df)*100:.0f}% do período).')
print(f'💡 Tendência de crescimento do D/E de 2015 para 2017 — alavancagem aumentando progressivamente.')
print(f'💡 Máximo registrado: D/E = {df["Debt_to_Equity"].max():.2f} — pico de risco relevante.')

## 7. Fluxo de Caixa e Dividendos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cash Flow mensal
cf_mensal = df.groupby('AnoMes').agg(
    CashFlow=('Cash_Flow', 'sum'),
    Dividendos=('Dividend_Payout', 'sum')
).reset_index()
cf_mensal['dt'] = cf_mensal['AnoMes'].dt.to_timestamp()

axes[0].bar(cf_mensal['dt'], cf_mensal['CashFlow'],
            color=[VERDE if v >= 0 else VERMELHO for v in cf_mensal['CashFlow']],
            edgecolor='white', width=20, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Fluxo de Caixa Mensal')
axes[0].set_ylabel('Cash Flow (R$)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[0].tick_params(axis='x', rotation=30)

# Dividendos vs Lucro por mês
lp_mensal = df.groupby('AnoMes').agg(
    Lucro=('Net_Income', 'sum'),
    Dividendos=('Dividend_Payout', 'sum')
).reset_index()
lp_mensal['dt'] = lp_mensal['AnoMes'].dt.to_timestamp()
lp_mensal['Payout_Ratio'] = lp_mensal['Dividendos'] / lp_mensal['Lucro'].clip(lower=1)

axes[1].plot(lp_mensal['dt'], lp_mensal['Lucro'],
             color=AZUL, linewidth=2, label='Lucro Líquido')
axes[1].plot(lp_mensal['dt'], lp_mensal['Dividendos'],
             color=LARANJA, linewidth=2, linestyle='--', label='Dividendos')
axes[1].set_title('Lucro Líquido vs Dividendos')
axes[1].set_ylabel('R$')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('img_caixa_dividendos.png', bbox_inches='tight')
plt.show()
print(f'💡 Cash Flow médio por registro: R$ {df["Cash_Flow"].mean():,.0f}')
print(f'💡 Dividendos pagos de forma estável (~R$ {df["Dividend_Payout"].mean():,.0f}/registro) independente do lucro — sinal de compromisso com acionistas.')

## 8. Correlações entre Variáveis

In [ ]:
cols_corr = [
    'Operating_Income', 'Expenses', 'Net_Income', 'Revenue',
    'Cash_Flow', 'ROA', 'Profit_Margin', 'Debt_to_Equity',
    'Dividend_Payout', 'Interest_Expense'
]
corr = df[cols_corr].corr()

# Máscara para triângulo superior
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlação de Pearson'})
ax.set_title('Matriz de Correlação — Indicadores Financeiros', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('img_correlacao.png', bbox_inches='tight')
plt.show()

# Destaques
corr_exp_ni = df['Expenses'].corr(df['Net_Income'])
corr_rev_ni = df['Revenue'].corr(df['Net_Income'])
print(f'💡 Correlação Expenses × Net_Income: {corr_exp_ni:.3f} — despesas têm impacto negativo relevante no lucro.')
print(f'💡 Correlação Revenue × Net_Income: {corr_rev_ni:.3f} — receita e lucro praticamente sem correlação linear,')
print(f'   o que sugere que variações de receita não se traduzem diretamente em lucro — eficiência de custos é o fator chave.')

## 9. Insights e Conclusões

| # | Indicador | Insight | Implicação |
|---|---|---|---|
| 1 | **Eficiência Operacional** | 19,5% dos registros com prejuízo operacional | Custos operacionais fora de controle em ~1 a cada 5 dias |
| 2 | **Margem de Lucro** | 18,9% com margem negativa | Alta volatilidade — lucro não é consistente |
| 3 | **ROA** | Média de 0.014 — muito baixo | Ativos elevados com retorno insuficiente |
| 4 | **Debt-to-Equity** | Média 5.53, crescendo ano a ano | Alavancagem elevada e em tendência de alta — risco sistêmico crescente |
| 5 | **Correlação custos × lucro** | -0.452 | Controle de despesas é o principal alavancador de rentabilidade |
| 6 | **Receita × Lucro** | Correlação de 0.004 | Crescer receita sem controlar custos não melhora resultado |
| 7 | **Dividendos** | Pagos de forma estável independente do lucro | Compromisso com acionistas — mas pode pressionar caixa em períodos negativos |

In [ ]:
print('='*58)
print('     RESUMO EXECUTIVO — DEUTSCHE BANK 2015–2017')
print('='*58)
print(f'Receita Total (2015–2016):  R$ {df[df["Ano"]<=2016]["Revenue"].sum():>15,.0f}')
print(f'Lucro Líquido (2015–2016):  R$ {df[df["Ano"]<=2016]["Net_Income"].sum():>15,.0f}')
print(f'ROA Médio:                       {df["ROA"].mean():>15.4f}')
print(f'Margem Média:                    {df["Profit_Margin"].mean():>15.4f}')
print(f'Debt-to-Equity Médio:            {df["Debt_to_Equity"].mean():>15.2f}')
print(f'Dias c/ Prejuízo Operacional: {df["Prejuizo_Op"].sum():>15} ({df["Prejuizo_Op"].mean()*100:.1f}%)')
print(f'Cash Flow Médio Diário:     R$ {df["Cash_Flow"].mean():>15,.0f}')
print('='*58)